# PMC — Exploration du nombre de neurones cachés

Ce notebook explore l'effet du nombre de neurones cachés sur la précision test.
Il complète `pmc.ipynb` où la meilleure config (16x16 couleur, epoch 28, lr=0.01) a été trouvée.


## 1. Imports


In [1]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt

PYTHON_DIR = os.path.abspath(os.path.join(os.getcwd(), '..', 'python'))
sys.path.insert(0, PYTHON_DIR)

from pmc import init, entrainer, precision_pmc
from functions import load_dataset, melanger

project_root = os.path.abspath('..')
train_folder = os.path.join(project_root, 'dataset', 'train_dataset')
test_folder  = os.path.join(project_root, 'dataset', 'test_dataset')

print('Imports OK')


Imports OK


## 2. Dataset


In [2]:
configs = {}
for resolution in (16, 32):
    for couleur in (False, True):
        X_train, Y_train = load_dataset(train_folder, target_size=(resolution, resolution), color=couleur, one_hot=True)
        X_test,  Y_test  = load_dataset(test_folder,  target_size=(resolution, resolution), color=couleur, one_hot=True)
        X_train, Y_train = melanger(X_train, Y_train, seed=42)
        configs[(resolution, couleur)] = (X_train, Y_train, X_test, Y_test)

for cle, (X_train, Y_train, X_test, Y_test) in configs.items():
    print(f'{cle}: train={len(X_train)}, test={len(X_test)}')

# Meilleure config trouvee dans pmc.ipynb
X_train, Y_train, X_test, Y_test = configs[(16, True)]
NB_CLASSES     = 3
NB_ENTREES     = 16 * 16 * 3  # 768
MEILLEUR_EPOCH = 28
MEILLEUR_LR    = 0.01


(16, False): train=2400, test=600
(16, True): train=2400, test=600
(32, False): train=2400, test=600
(32, True): train=2400, test=600


## 3. Exploration du nombre de neurones cachés

On teste de 2 à 16 neurones cachés et on garde le meilleur.


In [3]:
accs_cachees      = []
nb_cachees_values = list(range(2, 17))

for nb_cachees in nb_cachees_values:
    init(NB_ENTREES, nb_cachees, NB_CLASSES)
    entrainer(X_train, Y_train, epochs=MEILLEUR_EPOCH, alpha=MEILLEUR_LR)
    acc = precision_pmc(X_test, Y_test, NB_CLASSES)
    accs_cachees.append(acc)
    print(f'nb_cachees={nb_cachees} -> test={acc:.2f}%')

meilleur_cachees = nb_cachees_values[int(np.argmax(accs_cachees))]
print(f'Meilleur nb_cachees : {meilleur_cachees} -> {max(accs_cachees):.2f}%')


nb_cachees=2 -> test=47.33%
nb_cachees=3 -> test=45.67%
nb_cachees=4 -> test=47.83%
nb_cachees=5 -> test=45.67%
nb_cachees=6 -> test=48.50%
nb_cachees=7 -> test=47.83%
nb_cachees=8 -> test=50.33%
nb_cachees=9 -> test=48.50%
nb_cachees=10 -> test=47.83%
nb_cachees=11 -> test=49.83%
nb_cachees=12 -> test=49.67%
nb_cachees=13 -> test=49.50%
nb_cachees=14 -> test=49.33%
nb_cachees=15 -> test=48.17%
nb_cachees=16 -> test=50.00%
Meilleur nb_cachees : 8 -> 50.33%


## 4. Modèle final

Entraînement avec le meilleur nombre de neurones cachés trouvé.


In [4]:
init(NB_ENTREES, meilleur_cachees, NB_CLASSES)
entrainer(X_train, Y_train, epochs=MEILLEUR_EPOCH, alpha=MEILLEUR_LR)

acc_final = precision_pmc(X_test, Y_test, NB_CLASSES)
print(f'Modele final : nb_cachees={meilleur_cachees} -> test={acc_final:.2f}%')


Modele final : nb_cachees=8 -> test=50.33%
